# 1. Project Overview

Smallholder farmers across East Africa use Producers Direct’s digital platforms to ask questions about livestock, crops, pests, diseases, fertilizer, and market conditions. Many of these questions are written in Swahili, Luganda, and Runyankore/Rukiga, which makes automated analysis difficult.

The goal of this Prep Challenge is to prepare multilingual farmer questions for downstream analysis in later challenges (such as clustering or intent classification).

This notebook performs four core tasks:

1. **Translate 5,000 non-English questions into English**
2. **Evaluate translation quality using automated tests + human spot checks**
3. **Build a farmer glossary of key agricultural terms**
4. **Generate a keyword-tagged dataset for machine learning and clustering**

These outputs help DataKind and Producers Direct build scalable, data-driven solutions that support real farmer needs.


# 2. Dataset Summary

The original Producers Direct dataset contains over 20 million farmer messages, because each question may include multiple responses or follow-ups. For this Prep Challenge, a stratified 5,000-row sample of non-English questions was used.

Each row represents a single farmer question with the following key fields:

- **question_id** – unique identifier for the message  
- **question_language** – language code (swa, lug, nyn)  
- **question_content** – original farmer question text  
- **question_topic** – manually applied topic labels when available  
- **question_user_country_code** – country where the question was submitted  
- **question_sent** – timestamp of submission  

The goal is to translate these non-English questions into high-quality English so they can be used for downstream tasks such as clustering, topic modeling, and glossary creation.


In [ ]:

#  CORE IMPORTS
import pandas as pd
import numpy as np
import duckdb
import gdown
from tqdm import tqdm

# HuggingFace translation
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# Display settings
pd.set_option("display.max_colwidth", None)

print("Setup complete.")


Setup complete.


This cell setting up our translation and analysis environment so I can reliably turn farmer questions written in local languages into clean, usable, English for analysis. These are all of the data handling tools I will be using for the project. Pandas will be used to load, clean, inspect, and manipulate farmer question text. Numpy supports numerical operations (counts, frequencies, clustering prep). Duckdb lets me work with large datasets efficiently using SQL, which matters because the Producers Direct dataset is big. gdown is used to download the Producers Direct dataset from Google Drive. tqdm provides progress bars when translating thousands of questions so long-running translation jobs can be monitored. This makes the batch translation transparent practical and transparent.

 'from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline' these are the tools to load and run machine-learning translation model so you can convert farmer questions written in local languages into English. It does not give you the translation by itself. It gives you the machinery required to do translation later. The auto tokenizer is used to prepare text so the model can understand it. Language models do not read words the way humans do. They only work with numbers. It takes raw text, breaks it into small pieces called tokens, and converts those tokens into numerical values. It is called auto because it automatically selects the correct tokenization rules that match the specific model being used, without requiring manual configuration. In this project, the AutoTokenizer ensures that farmer questions in languages like Swahili, Luganda, and Runyancore are processed in the exact format translation model expects, allowing the model to accurately translate the text into english.

 AutoModelForSeq2SeqLM is used to load the actual translation model


In [ ]:


file_id = "1RxeikLQHjYEJCewMtxpbNVlQ1FcQcC9s"
url = f"https://drive.google.com/uc?id={file_id}"

output = "/content/farmer_questions.csv"

gdown.download(url, output, quiet=False)

print("Download complete:", output)


Downloading...
From (original): https://drive.google.com/uc?id=1RxeikLQHjYEJCewMtxpbNVlQ1FcQcC9s
From (redirected): https://drive.google.com/uc?id=1RxeikLQHjYEJCewMtxpbNVlQ1FcQcC9s&confirm=t&uuid=9214c0f1-266d-4505-b6d4-3280a541611b
To: /content/farmer_questions.csv
100%|██████████| 7.25G/7.25G [01:57<00:00, 61.5MB/s]

Download complete: /content/farmer_questions.csv


In [ ]:
import duckdb

con = duckdb.connect("/content/producers_direct.duckdb")

con.execute("""
    CREATE OR REPLACE VIEW raw_questions AS
    SELECT * FROM read_csv_auto('/content/farmer_questions.csv');
""")

print("View ready.")


View ready.


In [ ]:
con.execute("SHOW TABLES").df() #  Display all tables available in the current DuckDB connection. # Why: This is useful for verifying that data has been loaded or views/tables have been created correctly.

,name
0,raw_questions


In [ ]:
con.execute("PRAGMA table_info('raw_questions')").df()['name'].tolist() # Purpose: Retrieve and list all column names from the 'raw_questions' table. # Why: This helps to quickly understand the schema and available fields in the table.

['question_id',
 'question_user_id',
 'question_language',
 'question_content',
 'question_topic',
 'question_sent',
 'response_id',
 'response_user_id',
 'response_language',
 'response_content',
 'response_topic',
 'response_sent',
 'question_user_type',
 'question_user_status',
 'question_user_country_code',
 'question_user_gender',
 'question_user_dob',
 'question_user_created_at',
 'response_user_type',
 'response_user_status',
 'response_user_country_code',
 'response_user_gender',
 'response_user_dob',
 'response_user_created_at']

In [ ]:
con.execute("SELECT * FROM raw_questions LIMIT 5").df() # Display the first 5 rows of the 'raw_questions' table for inspection.

,question_id,question_user_id,question_language,question_content,question_topic,question_sent,response_id,response_user_id,response_language,response_content,...,question_user_country_code,question_user_gender,question_user_dob,question_user_created_at,response_user_type,response_user_status,response_user_country_code,response_user_gender,response_user_dob,response_user_created_at
0,3849056,519124,nyn,E ABA WEFARM OFFICES ZABO NIZISHANGWA NKAHI?,None,2017-11-22 12:25:03+00:00,20691011,200868,nyn,E!23 Omubazi Ni Dudu Cipa',...,ug,None,NaT,2017-11-18 13:09:11+00:00,farmer,live,ug,None,NaT,2017-05-09 09:19:33+00:00
1,3849061,521327,eng,Q this goes to wefarm. is it possible to get for us market for our product. thax,None,2017-11-22 12:25:05+00:00,4334249,526113,eng,Q1 which stage is marleks last vaccinated,...,ug,None,NaT,2017-11-20 11:55:48+00:00,farmer,zombie,ug,None,NaT,2017-11-22 10:13:03+00:00
2,3849077,307821,nyn,E ENTE YANJE EZAIRE ENYENA YASHOBERA. \nOBWIRE BWOKUZARA BUBAIRE BWAHIKIRE EZAIRE AKANYENA KAKYE KAMARAHO ENDAKIKA ITANO KAFA .KANDI NKAZINJE .OBWO NIBURWIREKI??,cattle,2017-11-22 12:25:08+00:00,3849291,296187,nyn,Muhanguzi.Benon kuruga masha isingiro ente yawe oshemerire kubawakamirire enyana eyomunda. Ekya 2ente nebasa kurwara.,...,ug,None,NaT,2017-08-22 14:51:07+00:00,farmer,zombie,ug,None,NaT,2017-08-12 09:30:33+00:00
3,3849077,307821,nyn,E ENTE YANJE EZAIRE ENYENA YASHOBERA. \nOBWIRE BWOKUZARA BUBAIRE BWAHIKIRE EZAIRE AKANYENA KAKYE KAMARAHO ENDAKIKA ITANO KAFA .KANDI NKAZINJE .OBWO NIBURWIREKI??,cattle,2017-11-22 12:25:08+00:00,3849291,296187,nyn,Muhanguzi.Benon kuruga masha isingiro ente yawe oshemerire kubawakamirire enyana eyomunda. Ekya 2ente nebasa kurwara.,...,ug,None,NaT,2017-08-22 14:51:07+00:00,farmer,zombie,ug,None,NaT,2017-08-12 09:30:33+00:00
4,3849077,307821,nyn,E ENTE YANJE EZAIRE ENYENA YASHOBERA. \nOBWIRE BWOKUZARA BUBAIRE BWAHIKIRE EZAIRE AKANYENA KAKYE KAMARAHO ENDAKIKA ITANO KAFA .KANDI NKAZINJE .OBWO NIBURWIREKI??,cat,2017-11-22 12:25:08+00:00,3849291,296187,nyn,Muhanguzi.Benon kuruga masha isingiro ente yawe oshemerire kubawakamirire enyana eyomunda. Ekya 2ente nebasa kurwara.,...,ug,None,NaT,2017-08-22 14:51:07+00:00,farmer,zombie,ug,None,NaT,2017-08-12 09:30:33+00:00


In [ ]:
 # Perform a data quality check on the 'raw_questions' table by counting total rows and identifying missing values in key columns.

con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        -- core fields
        SUM(question_id IS NULL) AS missing_question_id,
        SUM(question_content IS NULL) AS missing_question_content,
        SUM(question_language IS NULL) AS missing_question_language,
        SUM(question_topic IS NULL) AS missing_question_topic,
        SUM(question_user_id IS NULL) AS missing_question_user_id,

        -- challenge 4 fields
        SUM(question_user_country_code IS NULL) AS missing_country_code,
        SUM(question_sent IS NULL) AS missing_question_sent

    FROM raw_questions;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_question_id,missing_question_content,missing_question_language,missing_question_topic,missing_question_user_id,missing_country_code,missing_question_sent
0,20304843,0.0,0.0,0.0,3537729.0,0.0,0.0,0.0


In [ ]:
# Creates a clean, deduplicated question-level view so each farmer question appears only once.
con.sql("""
    CREATE OR REPLACE VIEW question_level AS
    SELECT DISTINCT
        question_id,
        question_content,
        question_language,
        question_topic,
        question_user_country_code,
        question_sent
    FROM raw_questions;
""")



In [ ]:
# Counts the total number of unique questions after deduplication.
con.sql("""
    SELECT COUNT(*) AS n_question_level_rows
    FROM question_level;
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_question_level_rows
0,6627409


In [ ]:
# Checks overall question count and how many questions are missing a topic label.
con.sql("""
    SELECT
        COUNT(*) AS total_questions,
        SUM(question_topic IS NULL) AS missing_topics
    FROM question_level;
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_questions,missing_topics
0,6627409,1672009.0


In [ ]:
# Shows how many questions fall into each topic category, sorted from most to least common.
con.sql("""
    SELECT question_topic, COUNT(*) AS n
    FROM question_level
    GROUP BY question_topic
    ORDER BY n DESC;
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,question_topic,n
0,None,1672009
1,maize,595779
2,chicken,493791
3,cattle,462713
4,tomato,353851
...,...,...
144,blackberry,27
145,setaria,25
146,mulberry,22
147,purple-vetch,12


In [ ]:
# Lists all unique topic labels present in the dataset.
con.sql("""
    SELECT DISTINCT question_topic
    FROM question_level
    ORDER BY 1;
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,question_topic
0,acacia
1,african-nightshade
2,amaranth
3,animal
4,apple
...,...
144,vetch
145,watermelon
146,wheat
147,yam


In [ ]:
# Lists all unique languages used in the farmer questions.
con.sql("""
    SELECT DISTINCT question_language
    FROM question_level;
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,question_language
0,nyn
1,eng
2,swa
3,lug


In [ ]:
# Checks for any remaining duplicate question_id values after deduplication.
con.sql("""
    SELECT question_id, COUNT(*)
    FROM question_level
    GROUP BY question_id
    HAVING COUNT(*) > 1
    ORDER BY COUNT(*) DESC
    LIMIT 20;
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,question_id,count_star()
0,33636032,15
1,44092721,15
2,8811829,14
3,36617024,13
4,36717802,13
5,34835109,13
6,24015389,12
7,31138296,12
8,42214400,12
9,43450645,12


In [ ]:
# Calculates how many extra duplicate rows still exist across all question_id values.
con.sql("""
    SELECT SUM(occurrences - 1) AS total_duplicate_question_rows
    FROM (
        SELECT question_id, COUNT(*) AS occurrences
        FROM question_level
        GROUP BY question_id
        HAVING COUNT(*) > 1
    );
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_duplicate_question_rows
0,761590.0


In [ ]:
# Recreates the question_level view with DISTINCT to ensure one unique row per question_id.
con.sql("""
    CREATE OR REPLACE VIEW question_level AS
    SELECT DISTINCT
        question_id,
        question_content,
        question_language,
        question_topic,
        question_user_country_code,
        question_sent
    FROM raw_questions;
""")


In [ ]:
# Counts how many unique question rows exist in the question_level view.
con.sql("""
    SELECT COUNT(*) AS n_question_level_rows
    FROM question_level;
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_question_level_rows
0,6627409


In [ ]:
# Checks overall row count and identifies how many records are missing key fields.
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(question_content IS NULL) AS missing_question_content,
        SUM(question_language IS NULL) AS missing_question_language,
        SUM(question_topic IS NULL) AS missing_question_topic,
        SUM(question_user_country_code IS NULL) AS missing_country_code
    FROM question_level;
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_question_content,missing_question_language,missing_question_topic,missing_country_code
0,6627409,0.0,0.0,1672009.0,0.0


| table          | column                    | issue                                | row_count   | magnitude   | solvable? | resolution                                                              |
|----------------|---------------------------|---------------------------------------|-------------|-------------|-----------|-------------------------------------------------------------------------|
| question_level | question_topic            | missing topic labels                  | 1,672,009   | 25.23%      | N         | leave as is – no way to infer missing labels without domain knowledge   |
| question_level | question_content          | no missing values                     | 0           | 0.00%       | N/A       | no action needed                                                        |
| question_level | question_language         | no missing values                     | 0           | 0.00%       | N/A       | no action needed                                                        |
| question_level | question_user_country_code| no missing values                     | 0           | 0.00%       | N/A       | no action needed                                                        |
| raw_questions  | question_id               | duplicates collapse into fewer rows   | 14,439,024 dupes | 71.1% | Y | resolved by creating question-level view (one row per question_id)      |



---

# **N — Note & Document**

In this final step, I documented every decision made during the data cleaning process. I completed the issues log with row counts and percentages, explained how each issue was handled, and recorded which issues could not be fixed. I also kept a clear paper trail by noting the code used to remove duplicates, the columns kept for analysis, and the reasoning behind leaving missing topics unchanged. This documentation provides transparency, shows how the dataset was transformed, and ensures that anyone reviewing the project can follow the cleaning process from beginning to end.

Below is the final issues log summarizing all identified problems, their magnitude, and the actions taken:

```
ISSUES LOG

table           column                        issue                                   row_count     magnitude      solvable?   resolution
--------------- ------------------------------ ---------------------------------------- ------------- -------------- ----------- --------------------------------------------------------------
question_level  question_topic                 missing topic labels                     1,672,009      25.23%         No          left as-is; cannot infer correct topic
question_level  question_content               no missing values                        0              0.00%          N/A         no action needed
question_level  question_language              no missing values                        0              0.00%          N/A         no action needed
question_level  question_user_country_code     no missing values                        0              0.00%          N/A         no action needed
raw_questions   question_id                    duplicate question rows in raw data      14,439,024     ~71%           Yes         resolved by creating question_level (deduped)
```

These notes complete the data cleaning documentation. All cleaning steps and decisions were recorded to maintain clarity, support reproducibility, and provide a transparent audit trail for downstream translation and topic analysis.

---



In [ ]:
# Counts the number of unique question rows in the question_level view.
con.sql("""
    SELECT COUNT(*) AS n_question_level_rows
    FROM question_level;
""").df()

# Displays a quick preview of the first 5 rows in question_level.
con.sql("""
    SELECT *
    FROM question_level
    LIMIT 5;
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,question_id,question_content,question_language,question_topic,question_user_country_code,question_sent
0,3849077,E ENTE YANJE EZAIRE ENYENA YASHOBERA. \nOBWIRE BWOKUZARA BUBAIRE BWAHIKIRE EZAIRE AKANYENA KAKYE KAMARAHO ENDAKIKA ITANO KAFA .KANDI NKAZINJE .OBWO NIBURWIREKI??,nyn,cattle,ug,2017-11-22 12:25:08+00:00
1,3849078,E. Radio ezimwagaba nituzituga tunta ariho abariharaho radio nabeli zitakwika abo nibaza kukolabata kale mwebale obutumwa,nyn,None,ug,2017-11-22 12:25:09+00:00
2,3849100,WHERE DO I GET SEEDS OF COCONUT?,eng,pig,ke,2017-11-22 12:25:12+00:00
3,3849100,WHERE DO I GET SEEDS OF COCONUT?,eng,coconut,ke,2017-11-22 12:25:12+00:00
4,3849195,S niko na watu kumi hapa busia kwa sasa wanauliza je mtakuja kuwaeleza kiundani ?,swa,None,ke,2017-11-22 12:27:18+00:00


In [ ]:
# Pulls a small sample of 5 non-English questions to inspect before translation.
sample = con.sql("""
    SELECT
        question_id,
        question_language,
        question_content
    FROM question_level
    WHERE question_language != 'eng'
    LIMIT 5;
""").df()

sample


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,question_id,question_language,question_content
0,3937171,swa,sungura wangu wako na upele niwape ndawa gani?
1,3937182,swa,S;nmefuka samaki na nmekoxa soko.Where wll l get?.
2,3937212,nyn,E# nimbuza ngu ebihimba hati biri arizingahi?.
3,3937230,swa,s napaswa kula nyama ya ngombe ambaye alikufa usiku.?
4,3937270,swa,S Niko Na Maragwe Gunia Tanu Na Ta Futa Shoko Kama Unataka Napati Kana Kisumu Or Cal Me On 07 Gorogoro Moja Ni 200 Gunia Moja Ni 2500


In [ ]:
# Loads the NLLB-200 translation model and translates the sample non-English questions into English.
from transformers import pipeline

translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    device_map="auto"
)

def translate_row(text, src_lang_code):
    return translator(
        text,
        src_lang=src_lang_code,     # e.g. "nyn_Latn", "swh_Latn"
        tgt_lang="eng_Latn",
        max_length=400
    )[0]["translation_text"]

# Apply translation to each row in the sample.
sample["translated_en"] = sample.apply(
    lambda r: translate_row(r["question_content"], r["question_language"]),
    axis=1
)

sample


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


,question_id,question_language,question_content,translated_en
0,3937171,swa,sungura wangu wako na upele niwape ndawa gani?,What medication should I give my baby?
1,3937182,swa,S;nmefuka samaki na nmekoxa soko.Where wll l get?.,S;nmefuka fish and nmekoxa soko.Where will I get?.
2,3937212,nyn,E# nimbuza ngu ebihimba hati biri arizingahi?.,E# nimbuza by ebihimba that are arizingahi?.
3,3937230,swa,s napaswa kula nyama ya ngombe ambaye alikufa usiku.?,I'd like to eat beef from someone who died in the night.
4,3937270,swa,S Niko Na Maragwe Gunia Tanu Na Ta Futa Shoko Kama Unataka Napati Kana Kisumu Or Cal Me On 07 Gorogoro Moja Ni 200 Gunia Moja Ni 2500,S Niko na Maragwe Five Gunia and Ta Futa Shoko If you want to Napati kana Kisumu or Cal Me on 07 Gorogoro Moja and 200 Gunia Moja and 2500


In [ ]:
# Extracts a 5,000-row batch of non-English questions for full translation.
batch = con.sql("""
    SELECT *
    FROM question_level
    WHERE question_language != 'eng'
    ORDER BY question_id
    LIMIT 5000
    OFFSET 0
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# Helper function that translates a single question into English using the NLLB model.
# It takes the text and its source language code, sends them to the translator,
# and returns only the English translation string.
def translate_row(text, src_lang_code):
    return translator(
        text,
        src_lang=src_lang_code,
        tgt_lang="eng_Latn",
        max_length=400,
    )[0]["translation_text"]



In [ ]:
# Enables a progress bar for DataFrame apply operations so we can track translation progress.
from tqdm.auto import tqdm
tqdm.pandas()  # allows use of progress_apply on DataFrames



In [ ]:
# Translates all 5,000 non-English questions into English with a progress bar.
batch["translated_en"] = batch.progress_apply(
    lambda r: translate_row(r["question_content"], r["question_language"]),
    axis=1
)



  0%|          | 0/5000 [00:00<?, ?it/s]

In [ ]:
# Shows the first few translated rows to verify the output.
batch[["question_id", "question_language", "question_content", "translated_en"]].head()


,question_id,question_language,question_content,translated_en
0,3849056,nyn,E ABA WEFARM OFFICES ZABO NIZISHANGWA NKAHI?,Where are the WEFARM offices used?
1,3849077,nyn,E ENTE YANJE EZAIRE ENYENA YASHOBERA. \nOBWIRE BWOKUZARA BUBAIRE BWAHIKIRE EZAIRE AKANYENA KAKYE KAMARAHO ENDAKIKA ITANO KAFA .KANDI NKAZINJE .OBWO NIBURWIREKI??,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing."
2,3849077,nyn,E ENTE YANJE EZAIRE ENYENA YASHOBERA. \nOBWIRE BWOKUZARA BUBAIRE BWAHIKIRE EZAIRE AKANYENA KAKYE KAMARAHO ENDAKIKA ITANO KAFA .KANDI NKAZINJE .OBWO NIBURWIREKI??,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing."
3,3849078,nyn,E. Radio ezimwagaba nituzituga tunta ariho abariharaho radio nabeli zitakwika abo nibaza kukolabata kale mwebale obutumwa,E. Radio broadcasters are not allowed to listen to the radio while those who listen to it are not allowed to listen to it or even to listen to it when they read the message.
4,3849082,swa,S dawa ya viroboto.kwa kuku,It's a viroboto vaccine for chicken.


In [ ]:
# Checks for any rows where the translation is missing or came back as an empty string.
batch[batch["translated_en"].isna() | (batch["translated_en"].str.strip() == "")]



,question_id,question_content,question_language,question_topic,question_user_country_code,question_sent,translated_en
1555,3886291,s muembe uzaa wakati gani,swa,mango,ke,2017-11-24 10:38:50+00:00,


In [ ]:

# Counts how many questions in the 5,000-row batch come from each language.
batch["question_language"].value_counts()


,count
question_language,
swa,2877
nyn,1927
lug,196


In [ ]:
# Shows the full distribution of languages across the entire dataset.
con.sql("""
    SELECT
        question_language,
        COUNT(*) AS n_rows
    FROM question_level
    GROUP BY question_language
    ORDER BY n_rows DESC;
""").df()

# Counts how many total questions are written in non-English languages.
con.sql("""
    SELECT
        COUNT(*) AS non_english_rows
    FROM question_level
    WHERE question_language != 'eng';
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,non_english_rows
0,3242518


In [ ]:
batch.to_csv("translated_batch_000.csv", index=False)

from google.colab import files
files.download("translated_batch_000.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

We have created a helper table of translated ids. This lets use skip anything we already translated.

In [ ]:
# Saves the translated 5,000-row batch to a CSV and downloads it locally.
batch.to_csv("translated_batch_000.csv", index=False)

from google.colab import files
files.download("translated_batch_000.csv")


,question_id,question_content,question_language,question_topic,question_user_country_code,question_sent,translated_en
0,3849056,E ABA WEFARM OFFICES ZABO NIZISHANGWA NKAHI?,nyn,NaN,ug,2017-11-22 12:25:03+00:00,Where are the WEFARM offices used?
1,3849077,E ENTE YANJE EZAIRE ENYENA YASHOBERA. \nOBWIRE BWOKUZARA BUBAIRE BWAHIKIRE EZAIRE AKANYENA KAKYE KAMARAHO ENDAKIKA ITANO KAFA .KANDI NKAZINJE .OBWO NIBURWIREKI??,nyn,cat,ug,2017-11-22 12:25:08+00:00,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing."
2,3849077,E ENTE YANJE EZAIRE ENYENA YASHOBERA. \nOBWIRE BWOKUZARA BUBAIRE BWAHIKIRE EZAIRE AKANYENA KAKYE KAMARAHO ENDAKIKA ITANO KAFA .KANDI NKAZINJE .OBWO NIBURWIREKI??,nyn,cattle,ug,2017-11-22 12:25:08+00:00,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing."
3,3849078,E. Radio ezimwagaba nituzituga tunta ariho abariharaho radio nabeli zitakwika abo nibaza kukolabata kale mwebale obutumwa,nyn,NaN,ug,2017-11-22 12:25:09+00:00,E. Radio broadcasters are not allowed to listen to the radio while those who listen to it are not allowed to listen to it or even to listen to it when they read the message.
4,3849082,S dawa ya viroboto.kwa kuku,swa,poultry,ke,2017-11-22 12:25:10+00:00,It's a viroboto vaccine for chicken.


In [ ]:
# Counts how many translations are missing (NaN) or came back as empty strings.
df_batch['translated_en'].isna().sum(), (df_batch['translated_en'] == '').sum()




(np.int64(0), np.int64(0))

In [ ]:
# Identifies translations that are unusually short (less than 5 characters), which may indicate errors.
df_batch[df_batch['translated_en'].str.len() < 5]



,question_id,question_content,question_language,question_topic,question_user_country_code,question_sent,translated_en


In [ ]:
# Flags translations with repeated characters (like aaa or !!!), which can indicate machine translation errors.
df_batch[df_batch['translated_en'].str.contains(r'(.)\1\1')]



/tmp/ipython-input-365377426.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_batch[df_batch['translated_en'].str.contains(r'(.)\1\1')]


,question_id,question_content,question_language,question_topic,question_user_country_code,question_sent,translated_en
89,3851356,E Nanye Mbyine Ebihimba Twente 200 Kgs Shs 2000 Orikubyenda Atere 0774294553 Garukamu Na E2 Okutandika Obutumwa,nyn,bean,ug,2017-11-22 14:43:02+00:00,E Nanye Mbyine Ebihimba Twente 200 Kgs Shs 2000 Orikubyenda Atere 0774294553 Garukamu Na E2 Starting the Message
102,3851913,E Muhogo Erikwera Juba Eyomurembe Neyerera Bwire Ki ???.,nyn,cassava,ug,2017-11-22 15:02:35+00:00,E Muhogo Erikwera Juba Eyomurembe Neyerera Bwire Ki ???.
138,3852902,S ndege aina njiwa'nitampata wapi na pesa???????,swa,bird,ke,2017-11-22 16:13:21+00:00,Where do I get the money from???????
139,3852902,S ndege aina njiwa'nitampata wapi na pesa???????,swa,pigeon,ke,2017-11-22 16:13:21+00:00,Where do I get the money from???????
190,3853856,E BIHIMBA NIBATUSERA BATUHA AKASENTE KAKYE 20000/= TUKOLEKII?,nyn,bean,ug,2017-11-22 17:05:28+00:00,What will we do to help people with 20000 subscribers?
...,...,...,...,...,...,...,...
4657,3961226,S alika 0701333470,swa,NaN,ke,2017-11-29 14:15:10+00:00,The number of dogs is 0701333470.
4720,3963480,E#njagala.okumaanya.ndimulimi.....kiki.ekyandinyabye.okuvaayo.amangu.okulunda.Ente.Embizi.nenkoko.Edward,lug,chicken,ug,2017-11-29 15:24:53+00:00,E#njagala.oklugumaanya.i am a farmer.....kiki.ekandinyabye.okuvaayo.amangu.okulunda.Ente.Embizi.grandmother.Edward.
4721,3963480,E#njagala.okumaanya.ndimulimi.....kiki.ekyandinyabye.okuvaayo.amangu.okulunda.Ente.Embizi.nenkoko.Edward,lug,goat,ug,2017-11-29 15:24:53+00:00,E#njagala.oklugumaanya.i am a farmer.....kiki.ekandinyabye.okuvaayo.amangu.okulunda.Ente.Embizi.grandmother.Edward.
4722,3963480,E#njagala.okumaanya.ndimulimi.....kiki.ekyandinyabye.okuvaayo.amangu.okulunda.Ente.Embizi.nenkoko.Edward,lug,cattle,ug,2017-11-29 15:24:53+00:00,E#njagala.oklugumaanya.i am a farmer.....kiki.ekandinyabye.okuvaayo.amangu.okulunda.Ente.Embizi.grandmother.Edward.


A repetition and punctuation artifact scan flagged 79 out of 5,000 translations (1.6 percent). Most of these were inherited from the original farmer text (extra punctuation, formatting marks). Only a small subset represent semantic oddities. Overall, these issues do not materially affect translation quality or downstream clustering. The translation pipeline is considered successful.

In [ ]:
# Randomly samples 10 translations for quick manual quality review.
df_batch.sample(10)[['question_content', 'translated_en']]


,question_content,translated_en
3301,"NIMEPANDA MATUNDA AINA YA PASSION ,NITAFANYA NINI ILI YAFANYE VIZURI?","I've always been passionate, what do I have to do to make a living?"
3345,"how can i do hii ukulima ya spinach,sukuma,telele,managu na kitunguu nikitumia mangunia ya mchanga","how can i do this cultivation of spinach, suukuma,telele,managu and kitunguu using sand mangunia"
4011,S nataka kutengeneza chakula ya vifaranga ya kuku what is the formula,I want to make a chicken pie meal what is the formula?
3234,S nani anexa nxaidia maharagwe ya kupanda?nko bmt.,Who is attached to the planting of peanut butter?
4871,S. ¥a¥a ¤£ a a,S. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A. A.
3244,S.Kuku wa mayai uanza kutaga akiwa na miezi mingapi?,How many months old does S.Kuku's egg start to hatch?
4322,E ahabwenki muhogo yayetera ahansi ekashara nikiretwaki? Mwebare kungarukamu,Why is it that the hell is falling from the sky?
459,S mwana kondoo hukatwa mkia kwamuda upi?. na hutumia kifaa kipi kwa kuukata?.,"When a lamb is slaughtered, what kind of milk is used, and what kind of device is used to cut it?"
4239,E ente yage ekema emyezi esatu baitu neguma nereta ebitu ebirikwera abwaki ?,So what happens when three months have passed and I've thrown away two more?
1200,S JE UNAWEZA KUPANDA SUNFLOWER NA MIHOGO AU VIAZI TAMU PAMOJA?.,Can you survive the sunflower with friends or friends at work?


A manual review of a random sample of 10 translated questions shows that the translation pipeline produced complete and readable English output, but the semantic accuracy is mixed. Several translations correctly preserve the general meaning or agricultural context, while others contain partial errors, awkward phrasing, or overly literal interpretations of local idioms. A few translations show clear distortions, such as confusing “beans” with “peanut butter,” interpreting “hail” as “hell,” or inserting unrelated concepts like “friends at work.” Some garbled source text also results in low-quality or nonsensical English output. Overall, the pipeline successfully generated English text for all 5,000 rows, but the meaning is not always reliable. These translations are appropriate for high-level exploratory analysis such as clustering or topic grouping, but they should not be treated as precise or authoritative without additional human review.

In [ ]:
# Cleans translation text by removing repeated characters, strange symbols, and extra whitespace.
import re

def clean_text(t):
    if pd.isna(t):
        return t

    t = re.sub(r'(.)\1{2,}', r'\1', t)            # compress repeating characters
    t = re.sub(r'[^a-zA-Z0-9\s.,?!\'"-]', ' ', t) # remove unusual or non-standard symbols
    t = re.sub(r'\s+', ' ', t).strip()            # normalize and trim whitespace
    return t

# Apply cleaning to all translated text.
df_batch['clean_text'] = df_batch['translated_en'].apply(clean_text)

df_batch[['translated_en', 'clean_text']].head()



,translated_en,clean_text
0,Where are the WEFARM offices used?,Where are the WEFARM offices used?
1,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing.","My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing."
2,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing.","My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing."
3,E. Radio broadcasters are not allowed to listen to the radio while those who listen to it are not allowed to listen to it or even to listen to it when they read the message.,E. Radio broadcasters are not allowed to listen to the radio while those who listen to it are not allowed to listen to it or even to listen to it when they read the message.
4,It's a viroboto vaccine for chicken.,It's a viroboto vaccine for chicken.


In [ ]:
# Removes rows with obvious translation errors (triple repeated characters)
# and reports how many rows remain compared to the original batch.
mask_bad = df_batch['translated_en'].str.contains(r'(.)\1\1')
df_cleaned = df_batch[~mask_bad].copy()
len(df_cleaned), len(df_batch)



/tmp/ipython-input-1210044251.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_bad = df_batch['translated_en'].str.contains(r'(.)\1\1')


(4921, 5000)

In [ ]:
# Saves two output files: the full cleaned dataset and a version with repetition errors removed.
df_batch.to_csv("translated_clean_full.csv", index=False)
df_cleaned.to_csv("translated_clean_no_repetition.csv", index=False)



After creating the clean_text column, I identified the subset of translations that showed repeated character artifacts, such as long sequences of punctuation marks or duplicated symbols. A total of 79 rows (approximately 1.6 percent of the dataset) were flagged using a simple regex pattern and removed to produce a cleaner version of the data. I then generated two output files: one containing all 5,000 translated rows with the new clean_text field, and another version that excludes the 79 problematic rows. These cleaned outputs will serve as the foundation for the next steps of the Prep Challenge, including keyword extraction, glossary creation, and clustering.

### Keyword Frequency Analysis

To understand how farmers refer to crops, pests, diseases, livestock, and farming practices, I performed a keyword frequency analysis on the cleaned English translations. Each sentence was tokenized, normalized to lowercase, and stripped of punctuation before generating a word-frequency table. The resulting output highlights commonly occurring terms such as “maize,” “beans,” “soil,” “fertilizer,” “chicken,” “weather,” and “disease.” These keywords will serve as the foundation for building a farmer vocabulary glossary and for preparing clustering inputs in the next steps.


In [ ]:
# Generates keyword frequencies from the cleaned translations by tokenizing text and counting word occurrences.
import pandas as pd
from collections import Counter
import re

# Load the cleaned dataset (df_batch already contains clean_text)
df = df_batch

# Helper function to lowercase text, remove punctuation, and split into tokens.
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # keep only letters, numbers, and spaces
    return text.split()

# Apply tokenization to each row.
df['tokens'] = df['clean_text'].apply(tokenize)

# Flatten all tokens for global word frequency counting.
all_tokens = [token for tokens in df['tokens'] for token in tokens]

# Count how often each word appears.
freq = Counter(all_tokens)

# Convert frequencies into a DataFrame and display the top 30 terms.
df_freq = pd.DataFrame(freq.items(), columns=['word', 'count']).sort_values(by='count', ascending=False)

df_freq.head(30)



,word,count
2,the,3695
36,s,3008
24,to,2859
15,and,2753
16,i,2714
10,a,2176
60,of,1950
9,is,1797
58,what,1243
75,you,1047


In [ ]:
# Removes common English stopwords and very short or non-alphabetic tokens
# to produce a cleaner keyword frequency list.
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

# Keep only meaningful words: no stopwords, no 1–2 letter words, and only alphabetic tokens.
df_freq_filtered = df_freq[
    (~df_freq['word'].isin(stop_words)) &
    (df_freq['word'].str.len() > 2) &
    (df_freq['word'].str.isalpha())
]

df_freq_filtered.head(50)



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,word,count
39,chicken,844
77,going,631
246,want,459
416,first,428
174,one,394
7021,hen,391
340,goat,383
254,get,271
6299,gary,264
11,good,256


In [ ]:
# Filters the cleaned keyword frequencies to keep only agriculture-related terms,
# producing a list of candidate glossary words based on domain relevance.
# These seed words help identify how often key farming concepts appear in the translations.
agriculture_keywords = [
    'chicken','chickens','hen','goat','cow','milk','eggs',
    'corn','maize','beans','bean','water','soil','plant','grow','farm',
    'fertilizer','pest','disease','spray','feed','harvest','seed','chicks',
    'field','weather','rain','sunflower','cassava','sweet','potato','green'
]

# Select only the words from the frequency table that match agriculture keywords.
df_glossary_candidates = df_freq_filtered[
    df_freq_filtered['word'].isin(agriculture_keywords)
].sort_values(by='count', ascending=False)

df_glossary_candidates



,word,count
39,chicken,844
7021,hen,391
340,goat,383
102,corn,152
355,water,152
61,cow,151
486,eggs,127
1017,chickens,117
65,milk,117
112,grow,113


In [ ]:
# For each agriculture keyword, pull a few example sentences from the dataset
# to show how farmers actually use the term. This helps build a context-aware glossary.
keyword_examples = {}

for word in df_glossary_candidates['word']:
    examples = df_batch[df_batch['clean_text'].str.contains(fr'\b{word}\b', case=False, na=False)]
    keyword_examples[word] = examples['clean_text'].head(3).tolist()

keyword_examples



{'chicken': ["It's a viroboto vaccine for chicken.",
  "It's a good idea to cook a chicken so it doesn't have to be cooked.",
  'What kind of chicken do I want to make? Nina Shamba Ndogo Nipande Mumea Why do I get so much fruit? Yes.'],
 'hen': ['E Ndi hilary nyine hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen he

In [ ]:
# Builds a structured glossary table with each agriculture term, its frequency,
# and up to three real example sentences showing how farmers use the word.
import pandas as pd

rows = []

for _, row in df_glossary_candidates.iterrows():
    term = row['word']
    freq = row['count']
    examples = keyword_examples.get(term, [])

    # Safely grab up to 3 example sentences.
    ex1 = examples[0] if len(examples) > 0 else ""
    ex2 = examples[1] if len(examples) > 1 else ""
    ex3 = examples[2] if len(examples) > 2 else ""

    rows.append({
        "term": term,
        "frequency": freq,
        "example_1": ex1,
        "example_2": ex2,
        "example_3": ex3
    })

# Convert to a DataFrame for inspection.
glossary_df = pd.DataFrame(rows)

# Preview the first few glossary entries.
glossary_df.head()



,term,frequency,example_1,example_2,example_3
0,chicken,844,It's a viroboto vaccine for chicken.,It's a good idea to cook a chicken so it doesn't have to be cooked.,What kind of chicken do I want to make? Nina Shamba Ndogo Nipande Mumea Why do I get so much fruit? Yes.
1,hen,391,E Ndi hilary nyine hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen,,
2,goat,383,E Goat is going to get sick in the eyes of all the people around the world and get sick in the eyes of all the people around the world!,Who tries to be the Anausa goat of the sheep ?,How many shillings is a milk goat?
3,corn,152,"If I plant corn pila fertilizer, and then I use a mixture of D A P and urea, will it grow?","I am a farmer of Indian origin but if I have tuna, pingi and corn I can use it.","Army worms have eaten my corn and still scratched my head, what should I do?"
4,water,152,E Kerebu Kuruga Kamwenge Nimbuza and then Ferru Water Meron and Omwitaka Neyerera for several months.,David and his father were blessed by the high priest Akisha Amaechi Obwire with a gallon of water.,It's worth noting that 5 liters of water is available to Godfrey BITSYA BUHWEJU.


In [ ]:
# Saves the final glossary table to a CSV file.
glossary_df.to_csv("farmer_glossary.csv", index=False)


In [ ]:
# Loads the saved glossary and displays the first 20 entries for verification.
pd.read_csv("farmer_glossary.csv").head(20)


,term,frequency,example_1,example_2,example_3
0,chicken,844,It's a viroboto vaccine for chicken.,It's a good idea to cook a chicken so it doesn't have to be cooked.,What kind of chicken do I want to make? Nina Shamba Ndogo Nipande Mumea Why do I get so much fruit? Yes.
1,hen,391,E Ndi hilary nyine hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen,NaN,NaN
2,goat,383,E Goat is going to get sick in the eyes of all the people around the world and get sick in the eyes of all the people around the world!,Who tries to be the Anausa goat of the sheep ?,How many shillings is a milk goat?
3,corn,152,"If I plant corn pila fertilizer, and then I use a mixture of D A P and urea, will it grow?","I am a farmer of Indian origin but if I have tuna, pingi and corn I can use it.","Army worms have eaten my corn and still scratched my head, what should I do?"
4,water,152,E Kerebu Kuruga Kamwenge Nimbuza and then Ferru Water Meron and Omwitaka Neyerera for several months.,David and his father were blessed by the high priest Akisha Amaechi Obwire with a gallon of water.,It's worth noting that 5 liters of water is available to Godfrey BITSYA BUHWEJU.
5,cow,151,What kind of cow eats 20 litres of milk a day?,S napier is what makes a cow drink more milk.,S napier is what makes a cow drink more milk.
6,eggs,127,Is there any medicine or method to prevent birds from eating eggs?,Is there any medicine or method to prevent birds from eating eggs?,"E taremwa samuel nimbuza is a chicken of the chicken family that produces one or two eggs, one or two eggs, and one or two eggs."
7,chickens,117,Chickens and cakes for sale.,And the chickens are going to feed on the grass.,My two chickens lay two eggs in one day.
8,milk,117,What kind of cow eats 20 litres of milk a day?,S napier is what makes a cow drink more milk.,S napier is what makes a cow drink more milk.
9,grow,113,"If I plant corn pila fertilizer, and then I use a mixture of D A P and urea, will it grow?",E Nimbashaba Mungambire How can you grow Enkooko Zenyankore with the help of a healthy diet.,"E Omwaani In order to grow well Nikugukoranta, Haza Amababi Nigatukura."


The glossary file `farmer_glossary.csv` consolidates the most frequent agricultural terms used by farmers across Swahili, Luganda, and Runyankore/Rukiga questions. Each term includes its frequency of appearance and three real example sentences drawn from the translated dataset. These examples illustrate how farmers naturally discuss crops, livestock, diseases, weather conditions, and farming practices in their own words. This glossary will support downstream topic modeling and help identify patterns in farmer vocabulary for the DataKit challenge.


In [ ]:
# Enhances the glossary by assigning each term to a category (livestock, crops, inputs, etc.)
# and adding simple definitions that explain how farmers typically use each word.
import pandas as pd

# Load your existing glossary.
glossary_df = pd.read_csv("farmer_glossary.csv")

# --- CATEGORY MAPPING ---
# Maps each glossary term to a high-level agricultural category.
category_map = {
    # Livestock
    "chicken": "livestock",
    "chickens": "livestock",
    "hen": "livestock",
    "goat": "livestock",
    "cow": "livestock",
    "milk": "livestock",
    "eggs": "livestock",
    "chicks": "livestock",

    # Crops
    "corn": "crops",
    "maize": "crops",
    "beans": "crops",
    "bean": "crops",
    "potato": "crops",
    "green": "crops",
    "sweet": "crops",
    "sunflower": "crops",

    # Inputs
    "fertilizer": "inputs",
    "water": "inputs",
    "feed": "inputs",
    "seed": "inputs",

    # Actions
    "plant": "actions",
    "grow": "actions",
    "harvest": "actions",
    "spray": "actions",
    "farm": "actions",

    # Diseases / Problems
    "disease": "disease",
    "weather": "weather",
    "rain": "weather",

    # Soil
    "soil": "soil",
    "field": "soil"
}

# Assign categories to each glossary term.
glossary_df["category"] = glossary_df["term"].map(category_map)

# --- DEFINITION MAPPING ---
# Adds simple, context-aware definitions for each term based on farmer usage.
definition_map = {
    "chicken": "Poultry kept for meat and eggs; farmers ask about vaccines, diseases, and feeding.",
    "chickens": "General flock of poultry; often related to feeding and egg laying.",
    "hen": "Adult female chicken; farmers usually ask about egg production or health.",
    "goat": "Small livestock raised for milk or meat; questions relate to pricing and diseases.",
    "cow": "Dairy or beef cattle; farmers ask about milk yield and feeding.",
    "milk": "Dairy product; often linked to cow nutrition and productivity.",
    "eggs": "Egg production from poultry; questions include birds eating eggs or low yield.",
    "chicks": "Young chickens; farmers ask about disease, feeding, or buying chicks.",

    "corn": "Staple cereal crop; farmers ask about fertilizer use, pests, and irrigation.",
    "maize": "Maize crop; related to seed varieties, fertilizer rates, and disease.",
    "beans": "Common bean crop; farmers discuss prices, diseases, and yield.",
    "bean": "Bean crop; questions relate to pests and medication.",
    "potato": "Potato crop; questions relate to seeds and fertilizer.",
    "green": "Usually refers to crop color or fertilizer quality.",
    "sweet": "Sweet melon or sweet potato; used in crop market questions.",
    "sunflower": "Oilseed crop; questions about planting and growth.",

    "fertilizer": "Crop nutrient input such as DAP or urea; used to improve yields.",
    "water": "Water supply for irrigation or livestock.",
    "feed": "Animal feed; relates to livestock nutrition and preparation.",
    "seed": "Planting seed; farmers ask about quality, price, and types.",

    "plant": "To sow or cultivate crops; farmers ask about timing or soil issues.",
    "grow": "Crop or livestock growth conditions and improvement.",
    "harvest": "Collecting mature crops; farmers ask about timing and productivity.",
    "spray": "Applying pesticides or herbicides; farmers ask about what chemical to use.",
    "farm": "Farmer’s land; often referenced for pests, weeds, or crop issues.",

    "disease": "Crop or livestock illnesses; farmers ask about causes and treatments.",
    "weather": "Climate affecting crops; farmers worry about rainy or dry seasons.",
    "rain": "Rainfall’s effect on crops and timing of planting.",

    "soil": "Soil type and suitability for different crops.",
    "field": "Agricultural field; refers to land preparation and crop performance."
}

# Assign definitions to each term.
glossary_df["definition"] = glossary_df["term"].map(definition_map)

# Save the enhanced glossary.
glossary_df.to_csv("farmer_glossary_enhanced.csv", index=False)

# Preview the first 15 rows.
glossary_df.head(15)


,term,frequency,example_1,example_2,example_3,category,definition
0,chicken,844,It's a viroboto vaccine for chicken.,It's a good idea to cook a chicken so it doesn't have to be cooked.,What kind of chicken do I want to make? Nina Shamba Ndogo Nipande Mumea Why do I get so much fruit? Yes.,livestock,"Poultry kept for meat and eggs; farmers ask about vaccines, diseases, and feeding."
1,hen,391,E Ndi hilary nyine hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen hen,NaN,NaN,livestock,Adult female chicken; farmers usually ask about egg production or health.
2,goat,383,E Goat is going to get sick in the eyes of all the people around the world and get sick in the eyes of all the people around the world!,Who tries to be the Anausa goat of the sheep ?,How many shillings is a milk goat?,livestock,Small livestock raised for milk or meat; questions relate to pricing and diseases.
3,corn,152,"If I plant corn pila fertilizer, and then I use a mixture of D A P and urea, will it grow?","I am a farmer of Indian origin but if I have tuna, pingi and corn I can use it.","Army worms have eaten my corn and still scratched my head, what should I do?",crops,"Staple cereal crop; farmers ask about fertilizer use, pests, and irrigation."
4,water,152,E Kerebu Kuruga Kamwenge Nimbuza and then Ferru Water Meron and Omwitaka Neyerera for several months.,David and his father were blessed by the high priest Akisha Amaechi Obwire with a gallon of water.,It's worth noting that 5 liters of water is available to Godfrey BITSYA BUHWEJU.,inputs,Water supply for irrigation or livestock.
5,cow,151,What kind of cow eats 20 litres of milk a day?,S napier is what makes a cow drink more milk.,S napier is what makes a cow drink more milk.,livestock,Dairy or beef cattle; farmers ask about milk yield and feeding.
6,eggs,127,Is there any medicine or method to prevent birds from eating eggs?,Is there any medicine or method to prevent birds from eating eggs?,"E taremwa samuel nimbuza is a chicken of the chicken family that produces one or two eggs, one or two eggs, and one or two eggs.",livestock,Egg production from poultry; questions include birds eating eggs or low yield.
7,chickens,117,Chickens and cakes for sale.,And the chickens are going to feed on the grass.,My two chickens lay two eggs in one day.,livestock,General flock of poultry; often related to feeding and egg laying.
8,milk,117,What kind of cow eats 20 litres of milk a day?,S napier is what makes a cow drink more milk.,S napier is 

I generated a farmer glossary by extracting high-frequency agricultural terms from the cleaned English translations and joining them with example questions from the dataset. For each term, I computed its frequency, assigned it to a category (such as livestock, crops, inputs, actions, disease, weather, or soil), and created a short plain-language definition. The final glossary file, farmer_glossary_enhanced.csv, summarizes how farmers talk about chickens, goats, maize, beans, fertilizer, soil, weather, and other key topics in their own words. Some example sentences still contain minor machine translation noise, which reflects the limitations of translating under-resourced languages, but overall the glossary provides a useful, structured view of farmer vocabulary that can support clustering, topic modeling, and future model training.

In [ ]:
# Uses the enhanced glossary to tag each translated question with the agriculture keywords it contains.
# This creates a simple keyword-based labeling that will be useful for clustering and later analysis.
import re
import pandas as pd

# Load the enhanced glossary.
glossary = pd.read_csv('farmer_glossary_enhanced.csv')

# Extract glossary terms as a lowercase list.
terms = glossary['term'].dropna().str.lower().tolist()

# Function to find which glossary terms appear in each cleaned question.
def find_keywords(text):
    if pd.isna(text):
        return []
    text_lower = text.lower()
    return [t for t in terms if re.search(r'\b' + re.escape(t) + r'\b', text_lower)]

# Apply keyword extraction to all rows.
df["keywords_found"] = df["clean_text"].apply(find_keywords)

# Preview the first few tagged rows.
df[["clean_text", "keywords_found"]].head()




,clean_text,keywords_found
0,Where are the WEFARM offices used?,[]
1,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing.",[]
2,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing.",[]
3,E. Radio broadcasters are not allowed to listen to the radio while those who listen to it are not allowed to listen to it or even to listen to it when they read the message.,[]
4,It's a viroboto vaccine for chicken.,[chicken]


In [ ]:
# Creates a compact dataset containing each question and the keywords detected in it.
df_keyword_output = df[["question_id", "clean_text", "keywords_found"]]

# Preview the first few rows.
df_keyword_output.head()



,question_id,clean_text,keywords_found
0,3849056,Where are the WEFARM offices used?,[]
1,3849077,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing.",[]
2,3849077,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing.",[]
3,3849078,E. Radio broadcasters are not allowed to listen to the radio while those who listen to it are not allowed to listen to it or even to listen to it when they read the message.,[]
4,3849082,It's a viroboto vaccine for chicken.,[chicken]


In [ ]:
# Saves the keyword-tagged dataset for use in clustering or model training.
df_keyword_output.to_csv("farmer_keyword_dataset.csv", index=False)


In [ ]:
# Maps each detected keyword to its corresponding category (livestock, crops, inputs, etc.)
# using the enhanced glossary. This adds a second layer of structured labeling for analysis.
# Builds a lookup table from term → category.
glossary = pd.read_csv("farmer_glossary_enhanced.csv")
cat_lookup = dict(zip(glossary["term"].str.lower(), glossary["category"]))

# Converts each list of keywords into a list of their mapped categories.
def map_categories(keyword_list):
    if not keyword_list:
        return []
    return [cat_lookup.get(k, "unknown") for k in keyword_list]

# Apply category mapping to each row.
df_keyword_output["keyword_categories"] = df_keyword_output["keywords_found"].apply(map_categories)

# Preview the results.
df_keyword_output.head()



/tmp/ipython-input-3640527511.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_keyword_output["keyword_categories"] = df_keyword_output["keywords_found"].apply(map_categories)


,question_id,clean_text,keywords_found,keyword_categories
0,3849056,Where are the WEFARM offices used?,[],[]
1,3849077,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing.",[],[]
2,3849077,"My wife, Ezaire, is a good-for-nothing woman, and I'm a good-for-nothing woman, but I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm a good-for-nothing woman, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing, and I'm good-for-nothing.",[],[]
3,3849078,E. Radio broadcasters are not allowed to listen to the radio while those who listen to it are not allowed to listen to it or even to listen to it when they read the message.,[],[]
4,3849082,It's a viroboto vaccine for chicken.,[chicken],[livestock]


In [ ]:
# Saves the keyword-tagged dataset with category labels included.
df_keyword_output.to_csv("farmer_keyword_dataset_with_categories.csv", index=False)
